In [3]:
import httpx
import time
import pandas as pd
import re
from transformers import pipeline
from datetime import datetime, timezone
import json
from pathlib import Path

/Users/mnatali/Projects/sentiment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
W0609 16:07:11.389000 57892 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
file_path = Path("/Users/mnatali/Projects/sentiment_analysis/brightdata_across_social_media_analysis/brightdata_social_exports/reddit_datacenters_posts.json")

with file_path.open("r", encoding="utf-8") as f:
    reddit_posts = json.load(f)

In [5]:
all_post_ids = []

env_post_ids = []
infr_post_ids = []
housing_post_ids = []
econ_post_ids = []
life_qual_post_ids = []
aesth_post_ids = []
gov_post_ids = []
not_useful_post_ids = []

environmental_keywords = ["environment", "pollution", "emissions", "diesel", "fumes", "light pollution", "water", "drought", "water consumption",  "waste", "carbon", "climate"]
infrastructure_utilities_keywords = ["utilities", "utility", "electric", "power", "grid", "blackout", "substation", "reliable", "reliability", "capacity", "infrastructure", "generator", "water"]
housing_keywords = ["housing", "house", "home", "rent", "property", "real estate", "house price", "housing price", "housing cost", "land cost", "land price", "gentrification"]
economic_keywords = ["job", "work", "revenue", "tax", "money", "rich", "salary", "economy", "investment", "business"]
life_quality_keywords = ["resident", "fumes", "noise", "loud", "quiet", "vibration", "light pollution", "sound", "smell", "traffic"]
aesthetics_keywords = ["landscape", "visual impact",  "aesthetics", "looks", "architecture", "design", "facade", "industrial", "windows", "dark"]
government_keywords = ["zoning", "zone", "planning", "approval", "permit", "city council", "bill", "mayor", "governor", "government"]

matched_posts = 0

for red_post in reddit_posts:
    post_id = red_post["post_id"]
    all_post_ids.append(post_id)
    text = (red_post["description"])

    matched = False

    if any(kw in text for kw in environmental_keywords):
        env_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in infrastructure_utilities_keywords):
        infr_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in housing_keywords):
        housing_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in economic_keywords):
        econ_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in life_quality_keywords):
        life_qual_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in aesthetics_keywords):
        aesth_post_ids.append(post_id)
        matched = True
    if any(kw in text for kw in government_keywords):
        gov_post_ids.append(post_id)
        matched = True
    if not matched:
        not_useful_post_ids.append(post_id)
    else:
        matched_posts += 1

print("Total posts:", len(all_post_ids))
print("Total posts with a theme:", matched_posts)
print("Found environmental posts:", len(env_post_ids))
print("Found infrastructure posts:", len(infr_post_ids))
print("Found housing posts:", len(housing_post_ids))
print("Found economic posts:", len(econ_post_ids))
print("Found life quality posts:", len(life_qual_post_ids))
print("Found aesthetic posts:", len(aesth_post_ids))
print("Found government posts:", len(gov_post_ids))

posts_by_id = {post["post_id"]: post for post in reddit_posts}

Total posts: 32900
Total posts with a theme: 30398
Found environmental posts: 9483
Found infrastructure posts: 20651
Found housing posts: 16309
Found economic posts: 23484
Found life quality posts: 13207
Found aesthetic posts: 10974
Found government posts: 12652


In [6]:
theme_lists = [env_post_ids, infr_post_ids, housing_post_ids, econ_post_ids, life_qual_post_ids, aesth_post_ids, gov_post_ids]
posts = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre"]

for theme in theme_lists:
    df1 = pd.DataFrame(columns=["ids", "text", "date", "upvotes", "number of comments", "subreddit", "environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"])
    post_ids = []
    post_texts = []
    post_dates = []
    post_num_comments = []
    post_upvotes = []
    post_subreddits = []


    for pid in theme:
        post_ids.append(pid)
        post = posts_by_id.get(pid)
        post_texts.append(post["description"])
        post_dates.append(post["date_posted"])
        post_upvotes.append(post["num_upvotes"])
        post_subreddits.append(post["community_name"])


    df1["ids"] = post_ids
    df1["text"] = post_texts
    df1["date"] = post_dates
    df1["upvotes"] = post_upvotes
    df1["subreddit"] = post_subreddits


    for col in ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]:
        df1[col] = False

    if theme == env_post_ids:
        df1["environment"] = True
    if theme == infr_post_ids:
        df1["infrastructure"] = True
    if theme == housing_post_ids:
        df1["housing"] = True
    if theme == econ_post_ids:
        df1["economy"] = True
    if theme == life_qual_post_ids:
        df1["life quality"] = True
    if theme == aesth_post_ids:
        df1["aesthetics"] = True
    if theme == gov_post_ids:
        df1["government"] = True
    posts = pd.concat([posts, df1], ignore_index=True)

len(posts['ids'])

106760

In [7]:
def remove_links(text):
    # Regex pattern to find typical URLs (http/https followed by non-whitespace characters)
    url_pattern = re.compile(r'\[https?://\S+\]|\(https?://\S+\)|\[www\.\S+\]|\(www\.\S+\)')
    # Replace found URLs with an empty string
    cleaned_text = url_pattern.sub('', text)
    return cleaned_text

sample_string = "The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...  [https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#](https://wjla.com/news/local/fairfax-county-considers-enhancing-data-center-guidelines-virginia-community-development-board-of-supervisors#)  A group of residents from the [Save Bren Mar From Data Center](https://sites.google.com/view/savebrenmar/home) group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial properties.  [https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board](https://patch.com/virginia/reston/data-center-zoning-policy-be-updated-fairfax-county-board)Fairfax County considers enhancing data center guidelines"
result = remove_links(sample_string)
print(result)

posts["text"] = posts["text"].apply(remove_links)

The board’s land use policy committee heard a presentation from the Department of Planning and Development about potential enhancements to data center use standards, including requiring a noise study and establishing a minimum distance from residential areas. The board had directed staff last May to provide research, findings and recommendations...  McKay said this process is designed to steer data centers into places that are most appropriate and make sure standards are set for environmental protections, noise mitigation and other issues that have people concerned. He said the county is learning from Loudoun and Prince William Counties, where data centers are becoming more and more prevalent...    A group of residents from the [Save Bren Mar From Data Center] group from the Mason District held up signs throughout Tuesday's legislative policy committee meeting, calling calling on the Fairfax county Supervisors to end by right data center development on commercial and industrial propert

In [8]:
grouping_cols = ["ids", "text", "date"]
posts = posts.groupby(grouping_cols, as_index=False).max()
theme_cols = ["environment", "infrastructure", "housing", "economy", "life quality", "aesthetics", "government"]
len(posts['ids'])

30398

In [9]:
posts.head()

,ids,text,date,upvotes,number of comments,subreddit,environment,infrastructure,housing,economy,life quality,aesthetics,government
0,t3_100yio2,"Being in Telecom since 1985, this is something...",2023-01-02T00:42:27.229Z,4,NaN,replika,False,False,True,True,False,False,False
1,t3_10254az,"According to a research report "" Neuromorphic ...",2023-01-03T10:55:58.213Z,1,NaN,MnMResearchStudies,False,True,True,True,True,True,True
2,t3_107pxv6,I have been working with PA support for a few ...,2023-01-09T21:03:30.581Z,11,NaN,paloaltonetworks,False,False,True,True,True,False,True
3,t3_107ubhz,We're looking to use more feature flags in our...,2023-01-09T23:49:15.993Z,10,NaN,webdev,False,True,True,False,False,False,False
4,t3_1087m2o,Riot wants to boast inclusion through characte...,2023-01-10T11:31:02.422Z,2928,NaN,VALORANT,True,True,False,True,False,True,False


In [35]:
# def convert_dates(utc_date):
#     date = datetime.fromtimestamp(utc_date, tz=timezone.utc)
#     return date

# posts['date'] = posts['date'].apply(convert_dates)
# posts.head(10)

In [38]:
classifier = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,      # <--- important
    padding=True,         # <--- for batching
    max_length=512        
)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [39]:
texts = posts["text"].astype(str).tolist()
results = classifier(texts, truncation=True, padding=True, max_length=512, batch_size=32)
posts["roberta"] = results

In [40]:
posts["name"] = posts["roberta"].apply(lambda x: x["label"])

def roberta_label(name):
    if name == 'LABEL_2':
        return "positive"
    elif name == 'LABEL_1':
        return "neutral"
    else:
        return "negative"
posts["label"] = posts["name"].apply(roberta_label)

posts["degree"] = posts["roberta"].apply(lambda x: x["score"])
posts = posts.drop(["roberta", "name"], axis=1)
posts.head(15)

,ids,text,date,votes,number of comments,environment,infrastructure,housing,economy,life quality,aesthetics,government,label,degree
0,10s4xtb,People who have left NOVA or planning to leave...,1.675382e+09,129,308.0,False,False,False,False,False,False,True,negative,0.544824
1,117zom7,Is staying in nova even worth it? I posted thi...,1.676973e+09,8,22.0,False,False,True,True,False,False,True,negative,0.649185
2,1341j3h,Anyone have HD or 4k (drone) footage of Ashbur...,1.682885e+09,0,8.0,False,False,False,False,True,False,True,negative,0.794359
3,14gh0c1,Proposed data center in Chantilly stirs up con...,1.687473e+09,4,7.0,True,True,True,False,False,False,False,negative,0.910441
4,152m2sz,Microsoft acquires 14-acre site in Sterling fo...,1.689648e+09,86,43.0,False,False,True,False,True,False,False,negative,0.906577
5,158wywd,Can someone tell me whats going on on barriste...,1.690257e+09,0,7.0,False,False,True,False,True,False,False,negative,0.690163
6,15cwyr6,Aren't the Loudon datacenters actually awesome...,1.690649e+09,413,246.0,False,True,True,True,True,False,False,negative,0.418970
7,171nn12,"Looking for cable tech jobs Hello, I currentl...",1.696626e+09,0,4.0,False,False,True,True,False,False,False,negative,0.537280
8,1al34nd,Need help with sources and talking points AGAI...,1.707313e+09,0,27.0,False,False,True,False,False,False,False,negative,0.721406
9,1bdp55a,Fairfax County considers enhancing data center...,1.710329e+09,6,7.0,True,False,True,False,True,True,True,negative,0.924399


In [41]:
# calculating average sentiment based on theme:
# Weigh all posts by their degree in the numerator and denominator, means that the average sentiment will just be +/- 1 if there are only positive or negative themes but other than that does a pretty good job of weighing neutrality
def avg_sentiment_calculation(theme):
    theme_posts = posts[posts[theme] == True]
    pos = theme_posts.loc[theme_posts["label"] == "positive", "degree"].sum()
    neg = theme_posts.loc[theme_posts["label"] == "negative", "degree"].sum()
    total = theme_posts["degree"].sum()
    return (pos-neg)/total

print("Average sentiment of environmental posts:", avg_sentiment_calculation("environment"))
print("Average sentiment of infrastructural posts:", avg_sentiment_calculation("infrastructure"))
print("Average sentiment of housing-related posts:", avg_sentiment_calculation("housing"))
print("Average sentiment of economic posts:", avg_sentiment_calculation("economy"))
print("Average sentiment of life-quality-related posts:", avg_sentiment_calculation("life quality"))
print("Average sentiment of aesthetics-related posts:", avg_sentiment_calculation("aesthetics"))
print("Average sentiment of governmental posts:", avg_sentiment_calculation("government"))

Average sentiment of environmental posts: -1.0
Average sentiment of infrastructural posts: -1.0
Average sentiment of housing-related posts: -1.0
Average sentiment of economic posts: -1.0
Average sentiment of life-quality-related posts: -1.0
Average sentiment of aesthetics-related posts: -1.0
Average sentiment of governmental posts: -1.0


In [42]:
## total average sentiment calculation

pos = posts.loc[posts["label"] == "positive", "degree"].sum()
neg = posts.loc[posts["label"] == "negative", "degree"].sum()
total = posts["degree"].sum()
print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


In [43]:
pos = posts.loc[posts["label"] == "positive"]
neg = posts.loc[posts["label"] == "negative"]
neutral = posts.loc[posts["label"] == "neutral"]

print("Percent of posts that are neutral:", len(neutral)/len(posts))
print("Percent of posts that are negative:", len(neg)/len(posts))
print("Percent of posts that are positive:", len(pos)/len(posts))

really_pos = pos.loc[posts["degree"] > 0.70]
really_neg = neg.loc[posts["degree"] > 0.70]

if len(neg) != 0:
    print("Percent of negative posts that are really negative:", len(really_neg)/len(neg))
if len(pos) != 0:
    print("Percent of positive posts that are really positive:", len(really_pos)/len(pos))

Percent of posts that are neutral: 0.0
Percent of posts that are negative: 1.0
Percent of posts that are positive: 0.0
Percent of negative posts that are really negative: 0.4957983193277311


In [44]:
def convert_dates(utc_date):
    date = datetime.fromtimestamp(utc_date, tz=timezone.utc)
    return date

In [45]:
pre_2010 = posts.loc[posts["date"] < 1262304000]

if len(pre_2010) == 0:
    print("None")
else:
    pre_2010['date'] = pre_2010['date'].apply(convert_dates)
    pre_2010.head(10)
    pos = pre_2010.loc[posts["label"] == "positive", "degree"].sum()
    neg = pre_2010.loc[posts["label"] == "negative", "degree"].sum()
    total = pre_2010["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

None


In [46]:
between_2010_2015 = posts.loc[(posts["date"] > 1262304000) & (posts["date"] < 1420070400)]

if len(between_2010_2015) == 0:
    print("None")
else:
    between_2010_2015['date'] = between_2010_2015['date'].apply(convert_dates)
    between_2010_2015.head(10)
    pos = between_2010_2015.loc[posts["label"] == "positive", "degree"].sum()
    neg = between_2010_2015.loc[posts["label"] == "negative", "degree"].sum()
    total = between_2010_2015["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

None


In [47]:
between_2015_2020 = posts.loc[(posts["date"] > 1420070400) & (posts["date"] < 1577836800)]

if len(between_2015_2020) == 0:
    print("None")
else:
    between_2015_2020['date'] = between_2015_2020['date'].apply(convert_dates)
    between_2015_2020.head(10)
    pos = between_2015_2020.loc[posts["label"] == "positive", "degree"].sum()
    neg = between_2015_2020.loc[posts["label"] == "negative", "degree"].sum()
    total = between_2015_2020["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_89670/2183586022.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  between_2015_2020['date'] = between_2015_2020['date'].apply(convert_dates)


In [48]:
between_2020_2025 = posts.loc[(posts["date"] > 1577836800) & (posts["date"] < 1735689600)]

if len(between_2020_2025) == 0:
    print("None")
else:
    between_2020_2025['date'] = between_2020_2025['date'].apply(convert_dates)
    between_2020_2025.head(10)
    pos = between_2020_2025.loc[posts["label"] == "positive", "degree"].sum()
    neg = between_2020_2025.loc[posts["label"] == "negative", "degree"].sum()
    total = between_2020_2025["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_89670/3710434052.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  between_2020_2025['date'] = between_2020_2025['date'].apply(convert_dates)


In [49]:
between_2025_2026 = posts.loc[posts["date"] > 1735689600]

if len(between_2025_2026) == 0:
    print("None")
else:
    between_2025_2026['date'] = between_2025_2026['date'].apply(convert_dates)
    between_2025_2026.head(10)
    pos = between_2025_2026.loc[posts["label"] == "positive", "degree"].sum()
    neg = between_2025_2026.loc[posts["label"] == "negative", "degree"].sum()
    total = between_2025_2026["degree"].sum()
    print("Average sentiment for the whole subreddit: ", (pos-neg)/total)

Average sentiment for the whole subreddit:  -1.0


/var/folders/9m/h28gbbc970j03ncf7v7dhqk80000gq/T/ipykernel_89670/1067228198.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  between_2025_2026['date'] = between_2025_2026['date'].apply(convert_dates)
